### Building a CNN for Object Detection using CoCo Dataset

In [55]:
import os
import torch
import cv2
import glob
import numpy as np
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw


In [56]:
import matplotlib.pyplot as plt

In [57]:
from sklearn.model_selection import train_test_split

In [58]:
import torch.nn as nn
import torch.nn.functional as F


In [59]:
import torch.optim as optim

In [60]:

# Load all image paths and labels
images = sorted(glob.glob("/Users/sadakakarla/Desktop/UB/Sem2/Computer Vision/Project/coco128/images/train2017/*.jpg"))
labels = sorted(glob.glob("/Users/sadakakarla/Desktop/UB/Sem2/Computer Vision/Project/coco128/labels/train2017/*.txt"))

# Train-validation split (90% train, 10% validation)
train_imgs, val_imgs, train_labels, val_labels = train_test_split(images, labels, test_size=0.1, random_state=42)

print(f"Train Images: {len(train_imgs)}")
print(f"Validation Images: {len(val_imgs)}")

Train Images: 115
Validation Images: 13


In [61]:
# Define a custom dataset class
class CoCo_dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
        
        
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert("RGB")
        
        # load bounding boxes
        
        label_path = self.labels[idx]
        boxes, labels = [], []  
        with open(label_path, "r") as f:
            for line in f.readlines():
                data = line.strip().split()
                class_id = int(data[0])
                x_center, y_center, width, height = map(float, data[1:])
                
                 # Convert YOLO format to (x1, y1, x2, y2)
                img_w, img_h = image.size
                x1 = (x_center - width / 2) * img_w
                y1 = (y_center - height / 2) * img_h
                x2 = (x_center + width / 2) * img_w
                y2 = (y_center + height / 2) * img_h

                boxes.append([x1, y1, x2, y2])
                labels.append(class_id)
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(boxes), torch.tensor(labels)

In [62]:
# Define a transform to resize and convert the images to tensor
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Create Train & Validation Datasets
train_dataset = CoCo_dataset(train_imgs, train_labels, transform=transform)
val_dataset = CoCo_dataset(val_imgs, val_labels, transform=transform)

# Separate data loaders for train & validation

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

Training samples: 115, Validation samples: 13


In [64]:
num_classes = 80  # COCO dataset has 80 classes

In [65]:
class CustomCNN_Object_Detection(nn.Module):
    def __init__(self, num_classes, num_predictions=10):
        super(CustomCNN_Object_Detection, self).__init__()
        
        self.num_predictions = num_predictions
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  
             
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128*28*28, 1024)
        self.fc2 = nn.Linear(1024, 512)
        
        # Output heads
        self.bbox_head = nn.Linear(512, num_predictions * 4)  # 4 coordinates per prediction
        self.class_head = nn.Linear(512, num_predictions * num_classes)  # num_classes per prediction
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool3(x)
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # Reshape outputs
        bounding_box = self.bbox_head(x).view(x.size(0), self.num_predictions, 4)  # Shape: [batch_size, num_predictions, 4]
        class_outputs = self.class_head(x).view(x.size(0), self.num_predictions, num_classes)  # Shape: [batch_size, num_predictions, num_classes]

        return bounding_box, class_outputs

In [66]:
model = CustomCNN_Object_Detection(num_classes)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CustomCNN_Object_Detection(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=100352, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=512, bias=True)
  (bbox_head): Linear(in_features=512, out_features=40, bias=True)
  (class_head): Linear(in_features=512, out_features=800, bias=True)
)

In [67]:
bounding_box_loss = nn.MSELoss()
class_loss = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [70]:
# Create Train & Validation Datasets
train_dataset = CoCo_dataset(train_imgs, train_labels, transform=transform)
val_dataset = CoCo_dataset(val_imgs, val_labels, transform=transform)

def collate_fn(batch):
    images, boxes, labels = zip(*batch)  # Unpack batch

    # Stack images (ensure all are same size)
    images = torch.stack(images, dim=0)

    # Pad boxes and labels to match the number of predictions (e.g., 10)
    max_objects = 10  # Match this to the model's num_predictions
    padded_boxes = []
    padded_labels = []

    for box, label in zip(boxes, labels):
        num_objects = len(box)

        # Pad boxes with zeros
        if num_objects < max_objects:
            pad_size = max_objects - num_objects
            padded_box = torch.cat([box, torch.zeros((pad_size, 4))], dim=0)  # Pad with zeros
            padded_label = torch.cat([label, torch.zeros((pad_size,), dtype=torch.long)], dim=0)  # Pad with zeros (background class)
        else:
            padded_box = box[:max_objects]  # Truncate if there are too many objects
            padded_label = label[:max_objects]

        padded_boxes.append(padded_box)
        padded_labels.append(padded_label)

    # Stack padded boxes and labels
    padded_boxes = torch.stack(padded_boxes, dim=0)
    padded_labels = torch.stack(padded_labels, dim=0)

    return images, padded_boxes, padded_labels

# Update DataLoaders to use `collate_fn`
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_fn)


In [71]:
def compute_loss(pred_boxes, pred_classes, gt_boxes, gt_labels):
    # pred_boxes: [batch_size, num_predictions, 4]
    # pred_classes: [batch_size, num_predictions, num_classes]
    # gt_boxes: [batch_size, num_predictions, 4]
    # gt_labels: [batch_size, num_predictions]

    # Create a mask to ignore padded values
    mask = (gt_labels != 0)  # Padded labels are 0 (background class)

    # Compute bounding box loss (only for valid objects)
    bbox_loss = F.mse_loss(pred_boxes[mask], gt_boxes[mask])

    # Compute classification loss (only for valid objects)
    class_loss = F.cross_entropy(pred_classes[mask], gt_labels[mask])

    return bbox_loss, class_loss

In [73]:
def compute_accuracy(pred_classes, gt_labels, mask):
    """
    Compute classification accuracy for valid objects (ignoring padded values).
    """
    # pred_classes: [batch_size, num_predictions, num_classes]
    # gt_labels: [batch_size, num_predictions]
    # mask: [batch_size, num_predictions] (True for valid objects)

    # Get predicted class labels
    pred_labels = torch.argmax(pred_classes[mask], dim=1)  # Shape: [num_valid_objects]

    # Compare with ground truth labels
    correct = (pred_labels == gt_labels[mask]).sum().item()
    total = mask.sum().item()

    return correct, total

In [77]:
total_correct = 0
total_samples = 0
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    for images, gt_boxes, gt_labels in train_loader:
        images = images.to(device)
        gt_boxes = gt_boxes.to(device)
        gt_labels = gt_labels.to(device)
        mask = (gt_labels != -1)

        optimizer.zero_grad()

        # Forward pass
        pred_boxes, pred_classes = model(images)

        # Compute loss
        bbox_loss, class_loss = compute_loss(pred_boxes, pred_classes, gt_boxes, gt_labels)
        loss = bbox_loss + class_loss

        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Compute accuracy
        correct, total = compute_accuracy(pred_classes, gt_labels, mask)
        total_correct += correct
        total_samples += total

        total_train_loss += loss.item()

    
        total_train_loss += loss.item()

    print(f"Epoch {epoch+1}: Train Loss: {total_train_loss:.4f}")
    # Print training loss and accuracy
    train_accuracy = 100 * total_correct / total_samples if total_samples > 0 else 0
    print(f"Epoch {epoch+1}: Train Loss: {total_train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")


Epoch 1: Train Loss: 657363.7578
Epoch 1: Train Loss: 657363.7578, Train Accuracy: 0.96%
Epoch 2: Train Loss: 670135.6641
Epoch 2: Train Loss: 670135.6641, Train Accuracy: 0.91%
Epoch 3: Train Loss: 649808.1816
Epoch 3: Train Loss: 649808.1816, Train Accuracy: 0.96%
Epoch 4: Train Loss: 635813.0586
Epoch 4: Train Loss: 635813.0586, Train Accuracy: 0.98%
Epoch 5: Train Loss: 617696.3594
Epoch 5: Train Loss: 617696.3594, Train Accuracy: 0.96%
Epoch 6: Train Loss: 620510.8867
Epoch 6: Train Loss: 620510.8867, Train Accuracy: 0.97%
Epoch 7: Train Loss: 589176.2930
Epoch 7: Train Loss: 589176.2930, Train Accuracy: 1.01%
Epoch 8: Train Loss: 625983.9121
Epoch 8: Train Loss: 625983.9121, Train Accuracy: 1.03%
Epoch 9: Train Loss: 598251.0059
Epoch 9: Train Loss: 598251.0059, Train Accuracy: 1.03%
Epoch 10: Train Loss: 575980.2344
Epoch 10: Train Loss: 575980.2344, Train Accuracy: 1.02%


In [85]:
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)